# KernelBench L1 23_Softmax
#### Payload type: single memory-bound GPU operator (row softmax), repeated forward passes

Source: [KernelBench/level1/23_Softmax.py](https://github.com/ScalingIntelligence/KernelBench/blob/main/KernelBench/level1/23_Softmax.py)

In [ ]:
%load_ext jumper_extension
%perfmonitor_fast_setup

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Problem sizes and input factories, verbatim from KernelBench
batch_size = 4096
dim = 393216

def get_inputs():
    x = torch.rand(batch_size, dim)
    return [x]

def get_init_inputs():
    return []  # No special initialization inputs needed

In [ ]:
torch.manual_seed(0)
inputs = [tensor.to(device) for tensor in get_inputs()]
iterations = 2000

In [ ]:
class Model(nn.Module):
    """
    Simple model that performs a Softmax activation.
    """
    def __init__(self):
        super(Model, self).__init__()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Applies Softmax activation to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, num_features).

        Returns:
            torch.Tensor: Output tensor with Softmax applied, same shape as input.
        """
        return torch.softmax(x, dim=1)


torch.manual_seed(0)
model = Model(*get_init_inputs()).to(device)
with torch.no_grad():
    for _ in range(iterations):
        output = model(*inputs)
    torch.cuda.synchronize()

In [ ]:
%perfmonitor_ai_review --benchmark --replay-mode full